# Italy MASE Demand Dashboard

Demand view of MASE *consumi petroliferi* (petroleum consumption), built from `scripts/update_italy.py` -> `data/processed/italy/italy_mase_consumption.parquet`.

## Sections

1. **Setup** — load parquet, headline product slice, kt -> kbd conversion
2. **Headline** — sum of non-overlapping reporting products (excl. petchem feedstock)
3. **View A — native headlines** — MASE reporting rows (BENZINA | AUTO TOTALE, etc.)
4. **View B — canonical** — Gasoline, Diesel, Jet fuel, ... rollup
5. **Recent trends** — last 24 months + MoM/YoY table
6. **YoY growth** — trailing 12 vs prior 12
7. **Seasonality index** — monthly index, last 5 complete years
8. **Seasonality by year** — `analytics.seasonality_by_year_chart` (canonical subplot titles; incl. preliminary)
9. **Interactive deep-dive** — product picker with provisional shading
10. **Fuel oil scope research** — scattered Fuel Oil components vs JODI `RESFUEL` composite
11. **MASE vs JODI** — cross-source comparison using `JODI_COMPARE_SERIES` (kbd)

## Conventions

- MASE native unit is **kt** (thousand tonnes, monthly flow). Charts use **kbd** via `analytics.units.convert_series` with per-product density.
- **Definitive** annual files cover 2002–2025; **preliminary** monthly files extend 2026+ (`is_provisional=True`, dashed).
- Headline charts use `reference.italy.DELIVERY_HEADLINE_NATIVE` (domestic delivery rows). Stock deltas, bunkers, and check totals stay in the parquet but are excluded from headline sums.
- MASE-vs-JODI panels use `reference.italy.JODI_COMPARE_SERIES` — Italy-specific composites (gasoil + bunker + stock delta; scattered fuel-oil Sub-category rows + bunker + stock delta).


## 1. Setup

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display


def _resolve_project_root() -> Path:
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / "scripts" / "update_italy.py").exists():
            return candidate
        if (candidate / "country_oil_scraper" / "scripts" / "update_italy.py").exists():
            return candidate / "country_oil_scraper"
    raise RuntimeError(f"Could not locate project root from cwd: {here}")


PROJECT_ROOT = _resolve_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from analytics.products import (
    CANONICAL_AGGREGATE_LABELS,
    CANONICAL_KIND_LABEL,
    SUBCATEGORY_TO_PRODUCT_KIND,
)
from analytics.units import convert_series
from reference.italy import (
    DELIVERY_HEADLINE_NATIVE,
    JODI_COMPARE_SERIES,
    REPORTING_PRODUCTS,
    compute_jodi_compare_kt,
    fuel_oil_jodi_native_labels,
)

PARQUET_PATH = PROJECT_ROOT / "data" / "processed" / "italy" / "italy_mase_consumption.parquet"

df = pd.read_parquet(PARQUET_PATH)
df["date"] = pd.to_datetime(df["date"])

demand = df[df["metric_type"] == "TOTDEMO"].copy()

# Headline native rows for charts (domestic delivery; excludes stock delta / bunkers).
HEADLINE_NATIVE = sorted(DELIVERY_HEADLINE_NATIVE)

# REPORTING_PRODUCTS keys -> analytics.units density kinds.
UNITS_KIND = {
    "gasoline": "gasoline",
    "jet_fuel": "jet",
    "diesel": "diesel",
    "gasoil_total": "diesel",
    "lpg": "lpg",
    "bitumen": "bitumen",
    "lubricants": "lubes",
    "fuel_oil": "fuel_oil",
    "other_kerosene": "kerosene",
    "naphtha_feedstock": "naphtha",
}
NATIVE_TO_KIND = {REPORTING_PRODUCTS[k]: UNITS_KIND[k] for k in REPORTING_PRODUCTS}
DISPLAY = {
    REPORTING_PRODUCTS[k]: CANONICAL_KIND_LABEL.get(UNITS_KIND[k], k.replace("_", " ").title())
    for k in REPORTING_PRODUCTS
}

# Non-overlapping domestic delivery rows for total-demand headline.
TOTAL_DEMAND_NATIVE = HEADLINE_NATIVE

# Multi-product charts: major end-use products (exclude gasoil_total overlap + feedstock).
CHART_PRODUCTS = [
    REPORTING_PRODUCTS[k]
    for k in (
        "gasoline",
        "jet_fuel",
        "diesel",
        "lpg",
        "fuel_oil",
        "bitumen",
        "lubricants",
        "other_kerosene",
    )
]

headline = demand[demand["product_native"].isin(HEADLINE_NATIVE)].copy()
headline["product_kind"] = headline["product_native"].map(NATIVE_TO_KIND)
headline["value_kbd"] = convert_series(
    headline["value"],
    "kt",
    "kbd",
    product_kind=headline["product_kind"],
    date=headline["date"],
)
PLOT_COL = "value_kbd"

# Canonical rollup from headline rows (Sub-category is already on parquet).
demand_canonical = (
    headline[headline["product_canonical"].notna()]
    .groupby(["date", "product_canonical", "is_provisional"], as_index=False)["value_kbd"]
    .sum()
)
demand_canonical["kind"] = demand_canonical["product_canonical"].map(SUBCATEGORY_TO_PRODUCT_KIND)
demand_canonical["panel"] = demand_canonical["kind"].map(CANONICAL_KIND_LABEL)

print(f"Loaded: {len(df):,} rows  ({df['date'].min().date()} -> {df['date'].max().date()})")
print(f"Demand (TOTDEMO): {len(demand):,} rows | headline products: {len(HEADLINE_NATIVE)}")
print(f"Provisional rows: {demand['is_provisional'].sum():,}")
print(f"Definitive through: {demand.loc[~demand['is_provisional'], 'date'].max().date()}")
print(f"Canonical panels: {sorted(demand_canonical['panel'].dropna().unique())}")


Loaded: 12,374 rows  (2002-01-01 -> 2026-04-01)
Demand (TOTDEMO): 12,374 rows | headline products: 10
Provisional rows: 144
Definitive through: 2025-12-01
Canonical panels: ['Bitumen', 'Diesel', 'Fuel oil', 'Gasoline', 'Jet fuel', 'Kerosene', 'LPG', 'Lubes & greases', 'Naphtha']


In [3]:
df.tail(60).to_csv('mase_italy.csv')

## 2. Headline — total petroleum demand

Sums non-overlapping MASE reporting rows. Petrochemical feedstock (`CARICA PETROLCHIMICA NETTA`) is excluded from the total.

In [ ]:
total = (
    headline[headline["product_native"].isin(TOTAL_DEMAND_NATIVE)]
    .groupby(["date", "is_provisional"], as_index=False)["value_kbd"]
    .sum()
    .sort_values("date")
)

fig = go.Figure()
obs = total[~total["is_provisional"]]
prov = total[total["is_provisional"]]
fig.add_trace(go.Scatter(x=obs["date"], y=obs["value_kbd"], mode="lines", name="Definitive"))
if not prov.empty:
    fig.add_trace(go.Scatter(
        x=prov["date"], y=prov["value_kbd"], mode="lines+markers",
        name="Preliminary", line=dict(dash="dash"),
    ))
roll = total["value_kbd"].rolling(12, min_periods=6).mean()
fig.add_trace(go.Scatter(
    x=total["date"], y=roll, mode="lines",
    name="12-month rolling avg", line=dict(dash="dot", width=2),
))
fig.update_layout(
    title="Italy petroleum consumption — headline total (kbd)",
    xaxis_title="Date", yaxis_title="kbd", height=440,
    hovermode="x unified", template="plotly_white",
)
fig.show()


## 3. View A — native headline products

In [ ]:
mp = headline[headline["product_native"].isin(CHART_PRODUCTS)].copy()
mp["label"] = mp["product_native"].map(DISPLAY)

fig = px.line(
    mp, x="date", y="value_kbd", color="label",
    title="Consumption by MASE headline row (kbd)",
    labels={"value_kbd": "kbd", "date": ""},
)
fig.update_layout(height=480, hovermode="x unified", template="plotly_white")
fig.show()


## 4. View B — canonical products

In [ ]:
fig = px.line(
    demand_canonical, x="date", y="value_kbd", color="panel",
    title="Consumption by canonical product (kbd)",
    labels={"value_kbd": "kbd", "panel": "Product"},
)
fig.update_layout(height=480, hovermode="x unified", template="plotly_white")
fig.show()


## 5. Recent trends (last 24 months)

In [ ]:
cutoff = headline["date"].max() - pd.DateOffset(months=23)
recent = headline[
    (headline["date"] >= cutoff) & (headline["product_native"].isin(CHART_PRODUCTS))
].copy()
recent["label"] = recent["product_native"].map(DISPLAY)

fig = px.line(recent, x="date", y="value_kbd", color="label", title="Last 24 months (kbd)")
fig.update_layout(height=420, template="plotly_white")
fig.show()

tbl = recent.sort_values(["label", "date"]).copy()
tbl["mom_pct"] = tbl.groupby("label")["value_kbd"].pct_change(periods=1) * 100
tbl["yoy_pct"] = tbl.groupby("label")["value_kbd"].pct_change(periods=12) * 100
latest = tbl.groupby("label").tail(1)[["date", "value_kbd", "mom_pct", "yoy_pct", "is_provisional"]]
display(latest.sort_values("value_kbd", ascending=False).round(1))


## 6. Year-over-year growth (trailing 12 vs prior 12)

In [ ]:
mp_full = headline[headline["product_native"].isin(CHART_PRODUCTS)].copy()
mp_full["label"] = mp_full["product_native"].map(DISPLAY)
last_date = mp_full["date"].max()
window_end = last_date
window_start = last_date - pd.DateOffset(months=11)
prior_end = window_start - pd.DateOffset(months=1)
prior_start = prior_end - pd.DateOffset(months=11)


def _mean_in_range(g, start, end):
    sl = g[(g["date"] >= start) & (g["date"] <= end)]
    return sl["value_kbd"].mean() if len(sl) else np.nan


rows = []
for label, g in mp_full.groupby("label"):
    cur = _mean_in_range(g, window_start, window_end)
    prev = _mean_in_range(g, prior_start, prior_end)
    yoy = (cur / prev - 1) * 100 if prev and prev > 0 else np.nan
    rows.append({"product": label, "trailing_12_avg_kbd": cur, "prior_12_avg_kbd": prev, "yoy_pct": yoy})
yoy_df = pd.DataFrame(rows).sort_values("yoy_pct")

fig = px.bar(
    yoy_df, x="yoy_pct", y="product", orientation="h",
    title=f"YoY change in trailing-12mo avg kbd (to {last_date:%Y-%m})",
    labels={"yoy_pct": "YoY %"},
)
fig.update_layout(height=360, template="plotly_white")
fig.show()
display(yoy_df.round(1))


## 7. Seasonality index (last 5 complete years)

In [ ]:
last_year = int(headline.loc[~headline["is_provisional"], "date"].dt.year.max())
years = list(range(last_year - 5, last_year))
mp_idx = headline[headline["product_native"].isin(CHART_PRODUCTS)].copy()
mp_idx["label"] = mp_idx["product_native"].map(DISPLAY)
mp_idx["year"] = mp_idx["date"].dt.year
mp_idx["month"] = mp_idx["date"].dt.month
mp_idx = mp_idx[mp_idx["year"].isin(years)]

annual = mp_idx.groupby(["label", "year"])["value_kbd"].mean().rename("annual_mean")
monthly = mp_idx.groupby(["label", "year", "month"])["value_kbd"].mean().reset_index()
monthly = monthly.merge(annual, on=["label", "year"])
monthly["index"] = 100 * monthly["value_kbd"] / monthly["annual_mean"]

fig = px.line(
    monthly, x="month", y="index", color="label", facet_col="label",
    facet_col_wrap=2, title="Seasonality index (100 = annual mean, last 5 definitive years)",
)
fig.update_xaxes(tickmode="linear", tick0=1, dtick=1)
fig.update_layout(height=900, template="plotly_white", showlegend=False)
fig.show()


## 8. Seasonality by calendar year

Includes **preliminary** months (e.g. 2026 YTD). By default the chart shows the last five calendar years plus the current year; click legend entries to reveal older years.

In [ ]:
from analytics import seasonality_by_year_chart

# Include provisional rows — they carry the timely market signal.
season_df = headline[headline["product_native"].isin(CHART_PRODUCTS)].copy()
current_year = int(season_df["date"].dt.year.max())
chart_labels = {p: DISPLAY[p] for p in CHART_PRODUCTS}

fig = seasonality_by_year_chart(
    season_df,
    products=CHART_PRODUCTS,
    product_col="product_native",
    value_col="value_kbd",
    product_labels=chart_labels,
    highlight_year=current_year,
    default_visible_prior_years=5,
    title="Seasonality by calendar year — Italy petroleum demand (kbd)",
    units_label="kbd",
)
fig.show()


## 9. Interactive product deep-dive

In [ ]:
def plot_product_deep_dive(product_native: str) -> None:
    sl = headline[headline["product_native"] == product_native].sort_values("date")
    if sl.empty:
        print(f"No data for {product_native!r}")
        return
    title = DISPLAY.get(product_native, product_native)
    fig = go.Figure()
    obs = sl[~sl["is_provisional"]]
    prov = sl[sl["is_provisional"]]
    fig.add_trace(go.Scatter(x=obs["date"], y=obs["value_kbd"], mode="lines", name="Definitive"))
    if not prov.empty:
        fig.add_trace(go.Scatter(
            x=prov["date"], y=prov["value_kbd"], mode="lines+markers",
            name="Preliminary", line=dict(dash="dash"),
        ))
    roll = sl["value_kbd"].rolling(12, min_periods=6).mean()
    fig.add_trace(go.Scatter(
        x=sl["date"], y=roll, mode="lines",
        name="12-month rolling avg", line=dict(dash="dot"),
    ))
    fig.update_layout(
        title=f"{title} — Italy MASE consumption (kbd)", height=400,
        template="plotly_white", hovermode="x unified",
    )
    fig.show()


picker = widgets.Dropdown(
    options=[(DISPLAY.get(p, p), p) for p in sorted(HEADLINE_NATIVE, key=lambda x: DISPLAY.get(x, x))],
    description="Product",
)
widgets.interact(plot_product_deep_dive, product_native=picker)


## 10. Fuel oil scope research

MASE fuel-oil demand is scattered across grade splits, bunkers, and consumer stock deltas. JODI reports a single `RESFUEL` balance category.

- **Domestic headline:** `OLIO COMB.LE | TOTALE` (delivery charts only)
- **JODI composite:** sum Fuel Oil **detail** rows when present, else headline, plus `BUNKERS | OLIO COMB.*` and `OLIO COMBUSTIBILE` stock delta — see `reference.italy.compute_fuel_oil_jodi_kt`
- **Grade splits:** A.T.Z., B.T.Z., termoelettrica, altri usi (definitive); do not add headline total when splits exist (avoids double-count)

In [ ]:
from reference.italy import (
    REPORTING_PRODUCTS,
    compute_fuel_oil_jodi_kt,
    fuel_oil_jodi_native_labels,
)

fo_groups = fuel_oil_jodi_native_labels()
FO_NATIVE = sorted(
    fo_groups["headline"]
    | fo_groups["details"]
    | fo_groups["bunker"]
    | fo_groups["stock_delta"]
)
FO_LABELS = {
    REPORTING_PRODUCTS["fuel_oil"]: "Fuel oil — headline total (domestic)",
    **{n: f"Fuel oil — detail ({n.split('|')[-1].strip()})" for n in fo_groups["details"]},
    **{n: "Bunkers — fuel oil" for n in fo_groups["bunker"]},
    **{n: "Stock delta — fuel oil" for n in fo_groups["stock_delta"]},
}

fo = demand[demand["product_native"].isin(FO_NATIVE)].copy()
fo["value_kbd"] = convert_series(
    fo["value"], "kt", "kbd", product_kind="fuel_oil", date=fo["date"]
)
fo["label"] = fo["product_native"].map(FO_LABELS)

fo_jodi = compute_fuel_oil_jodi_kt(demand)
fo_jodi["value_kbd"] = convert_series(
    fo_jodi["value_kt"], "kt", "kbd", product_kind="fuel_oil", date=fo_jodi["date"]
)
fo_jodi["label"] = "Fuel oil — JODI composite (scattered sum)"

plot_df = pd.concat(
    [
        fo[["date", "label", "value_kbd", "is_provisional"]],
        fo_jodi[["date", "label", "value_kbd", "is_provisional"]],
    ],
    ignore_index=True,
)

fig = px.line(
    plot_df, x="date", y="value_kbd", color="label",
    title="Italy fuel oil — MASE components vs JODI composite (kbd)",
    labels={"value_kbd": "kbd", "label": "Series"},
)
fig.update_layout(height=520, hovermode="x unified", template="plotly_white")
fig.show()

JODI_PARQUET = PROJECT_ROOT / "data" / "processed" / "jodi" / "jodi_secondary.parquet"
if JODI_PARQUET.exists():
    jodi = pd.read_parquet(JODI_PARQUET)
    jodi["date"] = pd.to_datetime(jodi["date"])
    jodi_fo = jodi[
        (jodi["ref_area"] == "IT")
        & (jodi["flow_breakdown"] == "TOTDEMO")
        & (jodi["unit_measure"] == "KBD")
        & (jodi["energy_product"] == "RESFUEL")
    ][["date", "obs_value"]].rename(columns={"obs_value": "value_kbd"})
    jodi_fo["label"] = "JODI — RESFUEL"

    compare_df = pd.concat(
        [
            fo_jodi[["date", "value_kbd"]].assign(label="MASE — JODI composite"),
            jodi_fo,
        ],
        ignore_index=True,
    )
    fig2 = px.line(
        compare_df, x="date", y="value_kbd", color="label",
        title="Fuel oil — MASE JODI composite vs JODI RESFUEL (kbd)",
    )
    fig2.update_layout(height=400, template="plotly_white", hovermode="x unified")
    fig2.show()

    cutoff = plot_df["date"].max() - pd.DateOffset(months=23)
    jodi_s = jodi_fo.set_index("date")["value_kbd"]
    rows = []
    for label in sorted(set(plot_df["label"])):
        s = plot_df.loc[plot_df["label"] == label].set_index("date")["value_kbd"]
        merged = pd.concat([s, jodi_s], axis=1, keys=["mase", "jodi"]).dropna()
        merged = merged.loc[merged.index >= cutoff]
        if merged.empty:
            continue
        gap = (merged["mase"] - merged["jodi"]).abs().mean()
        pct = gap / merged["jodi"].abs().mean() * 100
        rows.append({
            "mase_series": label,
            "mean_abs_gap_kbd": gap,
            "pct_of_jodi_level": pct,
            "mase_avg_kbd": merged["mase"].mean(),
            "jodi_avg_kbd": merged["jodi"].mean(),
        })
    display(pd.DataFrame(rows).sort_values("mean_abs_gap_kbd").round(1))
else:
    print(f"[skip] JODI parquet not found at {JODI_PARQUET}")

## 11. MASE vs JODI (TOTDEMO)

- **MASE:** `reference.italy.compute_jodi_compare_kt` per panel (kt → kbd)
- **JODI:** `ref_area == IT`, `flow_breakdown == TOTDEMO`, `unit_measure == KBD`
- **Gasoil:** composite = `GASOLIO | TOTALE GASOLI` + bunker gasoil + heating stock delta
- **Fuel oil:** scattered Fuel Oil sub-category rows + bunker + stock delta (section 10)


In [ ]:
from analytics import cross_source_comparison_chart

JODI_PARQUET = PROJECT_ROOT / "data" / "processed" / "jodi" / "jodi_secondary.parquet"
if not JODI_PARQUET.exists():
    print(f"[skip] JODI parquet not found at {JODI_PARQUET}")
    print("       Run python scripts/update_jodi.py first.")
else:
    jodi = pd.read_parquet(JODI_PARQUET)
    jodi["date"] = pd.to_datetime(jodi["date"])

    compare_keys = ["gasoline", "jet_fuel", "gasoil", "lpg", "fuel_oil"]
    mase_frames = []
    for key in compare_keys:
        spec = JODI_COMPARE_SERIES[key]
        kt = compute_jodi_compare_kt(demand, key)
        sl = kt.copy()
        sl["value_kbd"] = convert_series(
            sl["value_kt"],
            "kt",
            "kbd",
            product_kind=spec.product_kind,
            date=sl["date"],
        )
        sl["panel"] = spec.panel
        mase_frames.append(sl[["date", "panel", "value_kbd"]])

    mase_panel = (
        pd.concat(mase_frames, ignore_index=True)
        .groupby(["date", "panel"], as_index=False)["value_kbd"]
        .sum()
    )

    jodi_codes = [JODI_COMPARE_SERIES[k].jodi_energy_product for k in compare_keys]
    panel_by_jodi = {
        JODI_COMPARE_SERIES[k].jodi_energy_product: JODI_COMPARE_SERIES[k].panel
        for k in compare_keys
    }
    jodi_it = jodi[
        (jodi["ref_area"] == "IT")
        & (jodi["flow_breakdown"] == "TOTDEMO")
        & (jodi["unit_measure"] == "KBD")
        & (jodi["energy_product"].isin(jodi_codes))
    ].copy()
    jodi_it["panel"] = jodi_it["energy_product"].map(panel_by_jodi)
    jodi_it["value_kbd"] = jodi_it["obs_value"]

    panels = [JODI_COMPARE_SERIES[k].panel for k in compare_keys]
    fig = cross_source_comparison_chart(
        df_a=mase_panel,
        df_b=jodi_it,
        products=panels,
        product_col_a="panel",
        product_col_b="panel",
        value_col_a="value_kbd",
        value_col_b="value_kbd",
        label_a="MASE (Italy)",
        label_b="JODI",
        title="Italy TOTDEMO — MASE vs JODI (kbd)",
        units_label="kbd",
    )
    fig.show()

    cutoff_24 = mase_panel["date"].max() - pd.DateOffset(months=23)
    print("\nMean |gap| over last 24 months (kbd):")
    for panel in panels:
        m = mase_panel.loc[mase_panel["panel"] == panel].set_index("date")["value_kbd"]
        j = jodi_it.loc[jodi_it["panel"] == panel].set_index("date")["value_kbd"]
        merged = pd.concat([m, j], axis=1, keys=["mase", "jodi"]).dropna()
        merged = merged.loc[merged.index >= cutoff_24]
        if merged.empty:
            continue
        gap = (merged["mase"] - merged["jodi"]).abs().mean()
        pct = gap / merged["jodi"].abs().mean() * 100
        print(f"  {panel:16s}  mean|gap| = {gap:>8,.1f} kbd  ({pct:5.1f}% of JODI level)")
